# LLM Support Data Preparation

This notebook prepares the training and test datasets from the raw support tickets CSV.


## Imports

We only need pandas for data handling and scikit-learn for the stratified split.


In [ ]:
import pandas as pd
from prompt_utils import build_prompt
from project_paths import RAW_CSV_PATH, TRAIN_DATASET_PATH, TEST_DATASET_PATH, RAW_DIR, PROCESSED_DIR
from sklearn.model_selection import train_test_split

# Make dataframe previews easier to read in the notebook.
pd.set_option('display.max_colwidth', 120)


## Organize data folders

We keep the raw CSV in `data/raw/` and write the prepared outputs to `data/processed/`.


In [ ]:
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('Raw CSV path:', RAW_CSV_PATH)
print('Processed dir:', PROCESSED_DIR)


## Load the dataset

We load the CSV from `data/raw/` and inspect its structure before keeping only the allowed columns.


In [ ]:
df = pd.read_csv(RAW_CSV_PATH)

# Quick preview of the raw file.
df.head()


## Keep only authorized columns

The specification explicitly allows only these features as model inputs and target: `subject`, `body`, `queue`, `language`, and `business_type`.


In [ ]:
cols = ['subject', 'body', 'queue', 'language', 'business_type']
df = df[cols].copy()

# Remove incomplete rows so every example has the fields needed to build a prompt.
df = df.dropna(subset=cols).reset_index(drop=True)

df.info()


## Stratified train/test split

We split the dataset while preserving the class distribution of `queue` in both subsets.


In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    # Preserve the original class proportions of `queue` in both train and test.
    stratify=df['queue']
)

print('Train size:', len(train_df))
print('Test size:', len(test_df))

# Display the class distribution as percentages.
print('\nTrain class distribution:')
print((train_df['queue'].value_counts(normalize=True).sort_index() * 100).round(2).astype(str) + ' %')

print('\nTest class distribution:')
print((test_df['queue'].value_counts(normalize=True).sort_index() * 100).round(2).astype(str) + ' %')


## Build train and test datasets

We create a `prompt` column for the input and an `answer` column for the expected label.


In [ ]:
train_df = train_df.copy()
test_df = test_df.copy()

# Create independent working copies so we can safely add new columns without side effects.
# axis=1 means the function is applied row by row.
train_df['prompt'] = train_df.apply(
    lambda row: build_prompt(
        subject=row['subject'],
        body=row['body'],
        language=row['language'],
        business_type=row['business_type'],
    ),
    axis=1,
)
train_df['answer'] = train_df['queue']

test_df['prompt'] = test_df.apply(
    lambda row: build_prompt(
        subject=row['subject'],
        body=row['body'],
        language=row['language'],
        business_type=row['business_type'],
    ),
    axis=1,
)
test_df['answer'] = test_df['queue']

# Keep only the prompt and the expected label to create a clean training dataset.
train_dataset = train_df[['prompt', 'answer']].copy()
test_dataset = test_df[['prompt', 'answer']].copy()


## Inspect the prepared samples

A quick preview helps verify that the prompt format is consistent and readable.


In [ ]:
display(train_dataset.head(2))


In [ ]:
display(test_dataset.head(2))


## Export processed datasets

The prepared CSV files are written to `data/processed/` for reuse in the training notebook.


In [ ]:
# Save the prepared datasets for later use.
train_dataset.to_csv(TRAIN_DATASET_PATH, index=False)
test_dataset.to_csv(TEST_DATASET_PATH, index=False)
